In [42]:
import data.tromso_data as td
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots
import o2_fev1_analysis.plot_helpers as ph
import data.helpers as dh
import data.breathe_data as bd

In [47]:
dfBR = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

In [20]:
df = td.build_meas_df("T5")


*** Building O2 Saturation and FEV1 dataframe ***


In [4]:
def get_study_demographics(df):
    # df["BMI"] = df["Weight"] / (df["Height"] / 100) ** 2
    # print(f"Mean BMI: {df['BMI'].mean():.1f} ± {df['BMI'].std():.1f}")
    print(f"Mean Age: {df['Age'].mean():.1f} ± {df['Age'].std():.1f}")
    n_female = (df["Sex"] == "Female").sum()
    print(f"Female: {n_female} ({n_female/len(df)*100:.1f}%)")

    n_real = (
        df.ID.nunique()
        - df.groupby("ID").apply(lambda df: df["FEV1 % Predicted"].max()).isna().sum()
    )
    ppfev1 = df.groupby("ID").apply(lambda df: df["FEV1 % Predicted"].max())
    n_id_sup_90 = len(ppfev1[ppfev1 >= 90])
    prct_id_sup_90 = n_id_sup_90 / n_real * 100
    n_id_70_90 = len(ppfev1[(ppfev1 >= 70) & (ppfev1 < 90)])
    prct_id_70_90 = n_id_70_90 / n_real * 100
    n_id_40_70 = len(ppfev1[(ppfev1 >= 40) & (ppfev1 < 70)])
    prct_id_40_70 = n_id_40_70 / n_real * 100
    n_id_inf_40 = len(ppfev1[ppfev1 < 40])
    prct_id_inf_40 = n_id_inf_40 / n_real * 100

    print(
        f"ppFEV1: {df['FEV1 % Predicted'].mean():.1f}% ± {df['FEV1 % Predicted'].std():.1f}%"
    )
    print("ppFEV1 subgrouping")
    print(f"  >=90%: {n_id_sup_90} ({prct_id_sup_90:.0f}%)")
    print(f"  70-89%: {n_id_70_90} ({prct_id_70_90:.0f}%)")
    print(f"  40-69%: {n_id_40_70} ({prct_id_40_70:.0f}%)")
    print(f"  <40%: {n_id_inf_40} ({prct_id_inf_40:.0f}%)")
    print(f"  Total: {n_real}")
    return -1


get_study_demographics(df)

Mean Age: 65.7 ± 9.4
Female: 2865 (56.1%)
ppFEV1: 87.4% ± 18.0%
ppFEV1 subgrouping
  >=90%: 2421 (47%)
  70-89%: 1899 (37%)
  40-69%: 719 (14%)
  <40%: 66 (1%)
  Total: 5105


-1

In [21]:
df

,ID,UID,Age,Sex,Height,Health,Asthma,Bronchitis,Smoke daily,Cigarettes number,...,Chest wheezing,Short winded walking fast,Short winded resting,O2 Saturation,FEV1,FEF2575,Predicted FEV1,Healthy O2 Saturation,FEV1 % Predicted,O2 Saturation % Healthy
0,1,2,60,Male,163.4,3.0,0.0,0.0,2.0,8,...,0.0,0.0,0.0,97.00,3.182,1.235,2.975139,97.322299,106.952987,99.668833
1,2,3,57,Male,185.1,4.0,0.0,0.0,1.0,10,...,0.0,1.0,0.0,94.67,3.703,3.041,4.043786,96.933067,91.572591,97.665330
2,3,5,56,Male,177.1,3.0,0.0,0.0,2.0,15,...,0.0,1.0,0.0,97.00,2.836,3.172,3.701227,97.076563,76.623238,99.921132
3,4,13,73,Male,170.2,2.0,0.0,0.0,2.0,NaN,...,0.0,0.0,0.0,98.33,2.370,1.231,2.821669,97.200328,83.992850,101.162210
4,5,14,70,Male,176.2,3.0,1.0,0.0,2.0,7,...,NaN,0.0,0.0,94.67,2.613,1.057,3.151908,97.092706,82.902165,97.504750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5100,7446,20947,76,Female,157.3,3.0,0.0,0.0,2.0,10,...,1.0,1.0,0.0,90.00,0.598,0.298,1.901237,98.252667,31.453209,91.600567
5101,7447,20956,80,Female,153.9,2.0,0.0,0.0,3.0,NaN,...,NaN,NaN,NaN,97.67,1.138,1.246,1.720913,98.313653,66.127673,99.345307
5102,7448,20957,80,Male,166.9,2.0,0.0,0.0,1.0,2,...,1.0,1.0,0.0,97.00,1.753,0.937,2.501434,97.259520,70.079805,99.733168
5103,7449,20958,78,Female,159.3,2.0,0.0,0.0,1.0,8,...,NaN,NaN,NaN,97.33,1.516,0.270,1.901240,98.216793,79.737429,99.097107


In [72]:
def plot_o2_fev_with_displots(O2_FEV1, x, y, title, rangex, rangey):

    opacity_scatter = 0.6
    opacity_displot = 1

    fig = make_subplots(
        rows=2,
        cols=2,
        shared_xaxes=True,
        shared_yaxes=True,
        column_widths=[0.8, 0.2],
        row_heights=[0.3, 0.7],
        vertical_spacing=0.02,
        horizontal_spacing=0.005,
    )

    x_val = O2_FEV1[x]
    y_val = O2_FEV1[y]

    # Add scatter plot
    fig.add_trace(
        go.Scatter(
            x=x_val,
            y=y_val,
            mode="markers",
            # name="Stable",
            marker=dict(
                size=5,
                color=ph.get_stable_color(opacity_scatter),
                line=dict(width=0.2, color="DarkSlateGrey"),
            ),
        ),
        row=2,
        col=1,
    )
    fig.update_xaxes(range=rangex, row=2, col=1)
    fig.update_yaxes(range=rangey, row=2, col=1)

    # Add displot for x
    if np.mean(x_val) > 4.5:
        # x_bins = dict(start=15, end=110, size=5)
        x_bins = dict(start=rangex[0], end=rangex[1], size=5)
    else:
        x_bins = dict(start=0.4, end=4.5, size=0.2)

    fig.add_trace(
        go.Histogram(
            x=x_val,
            histnorm="probability",
            # nbinsx=bins_n_stable,
            xbins=x_bins,
            marker=dict(color=ph.get_stable_color(opacity_displot)),
        ),
        row=1,
        col=1,
    )
    y_bins = dict(start=np.floor(rangey[0] * 0.9), end=np.ceil(rangey[1] * 1.1), size=1)
    fig.add_trace(
        go.Histogram(
            y=y_val,
            histnorm="probability",
            # Number of bins automatically set is good enough because O2 saturation is a dicsrete variable with a small values span
            ybins=y_bins,
            marker=dict(color=ph.get_stable_color(opacity_displot)),
        ),
        row=2,
        col=2,
    )

    # fig.update_layout(barmode="overlay")
    fig.update_xaxes(title_text=x, row=2, col=1)
    fig.update_yaxes(title_text=y, row=2, col=1)

    fig.update_layout(height=600, width=1300, title=title)

    return fig

In [82]:
title = f"Tromso T5 study, {df.shape[0]} entries"
xcol = "FEV1 % Predicted"
rangex=[np.floor(min(df[xcol].min(), dfBR[xcol].min()))*0.9, np.ceil(max(df[xcol].max(), dfBR[xcol].max()))*1.05]
ycol = "O2 Saturation"
rangey=[np.floor(min(df[ycol].min(), dfBR[ycol].min()))*0.99, np.ceil(max(df[ycol].max(), dfBR[ycol].max()))*1.01]
print(rangex, rangey)
fig = plot_o2_fev_with_displots(df, "FEV1 % Predicted", "O2 Saturation", title, rangex, rangey)
fig.write_image(f"{dh.get_path_to_main()}/PlotsCrossStudies/{title}.pdf")
# fig.show()

[4.5, 156.45000000000002] [74.25, 101.0]


In [83]:
title = f"Project Breathe study, {dfBR.shape[0]} entries"
fig = plot_o2_fev_with_displots(dfBR, "FEV1 % Predicted", "O2 Saturation", title, rangex, rangey)
fig.write_image(f"{dh.get_path_to_main()}/PlotsCrossStudies/{title}.pdf")
# fig.show()